In [ ]:
from experiments._common import H4_FCIDUMP_PATH, H6_FCIDUMP_PATH

from qiskit.quantum_info import SparsePauliOp

from qcmps import (
    FCIDumpLoader,
    GlobalQCMPS, Ansatz, HFPreset,
    Logger,                      
    GlobalOptimizer, SweepOptimizer,
    SO4Gate, BrickWallLayerType,
)

In [2]:
logging_path = "h4.log"
logger = Logger("h4.log")

In [3]:
loader = FCIDumpLoader(H4_FCIDUMP_PATH)
loader.interleave_spin_orbitals()

nuclear_repulsion = loader.nuclear_repulsion

hf_occupancy = loader.hf_occupancy()

num_phys_qubits = loader.n_spin_orbitals
num_bond_qubits = 2
num_qubits = num_phys_qubits + num_bond_qubits

full_hamiltonian = SparsePauliOp.from_list([
    ("I" * num_bond_qubits + label, coefficient)
    for label, coefficient in loader.hamiltonian.to_list()
])

In [4]:
ansatz = Ansatz(num_qubits)

hf_preset = HFPreset(num_phys_qubits, hf_occupancy)
bw_layer = BrickWallLayerType(SO4Gate)
qcmps = GlobalQCMPS(num_phys_qubits, num_bond_qubits, [bw_layer])

ansatz.add_component(hf_preset, list(range(num_bond_qubits, num_qubits)))
ansatz.add_component(qcmps, list(range(num_qubits)))

ansatz.add_hamiltonian(full_hamiltonian, list(range(num_qubits)))
ansatz.add_nuclear_repulsion(nuclear_repulsion)

In [5]:
logger.log(f"HF energy: {ansatz.total_energy():.12f}")
qcmps.uniform_parameters()
logger.log(f"Initial energy: {ansatz.total_energy():.12f}")

HF energy: -2.215215410025
Initial energy: -0.191030419094


In [9]:
logger.log("Starting QCMPS optimization...")
global_optimizer = GlobalOptimizer(logger)
result = global_optimizer.optimize(ansatz, qcmps, options = {"maxiter": 100, "gtol": 1e-6})

logger.log(f"Final energy: {ansatz.total_energy():.12f}")

Starting QCMPS optimization...
[Optimizer] Total Energy: -2.221006126274
[Optimizer] Total Energy: -2.221009924201
[Optimizer] Total Energy: -2.221013122855
[Optimizer] Total Energy: -2.221018987540
[Optimizer] Total Energy: -2.221030735375
[Optimizer] Total Energy: -2.221052398622
[Optimizer] Total Energy: -2.221093272916
[Optimizer] Total Energy: -2.221146970524
[Optimizer] Total Energy: -2.221219821646
[Optimizer] Total Energy: -2.221391462987
[Optimizer] Total Energy: -2.221560996533
[Optimizer] Total Energy: -2.221725408639
[Optimizer] Total Energy: -2.221873326407
[Optimizer] Total Energy: -2.221922367402
[Optimizer] Total Energy: -2.221952485998
[Optimizer] Total Energy: -2.222023089196
[Optimizer] Total Energy: -2.222108251937
[Optimizer] Total Energy: -2.222179289475
[Optimizer] Total Energy: -2.222222727378
[Optimizer] Total Energy: -2.222267762412
[Optimizer] Total Energy: -2.222302150974
[Optimizer] Total Energy: -2.222358276827
[Optimizer] Total Energy: -2.222468704255
[Op

In [7]:
logger.log(f"Adding additional layers to QCMPS...")
qcmps.add_layers([bw_layer])
ansatz.rebuild()
logger.log(f"Energy after adding layers: {ansatz.total_energy():.12f}")
qcmps.perturb_parameters(0.001)
logger.log(f"Energy after perturbing parameters: {ansatz.total_energy():.12f}")

Adding additional layers to QCMPS...
Energy after adding layers: -2.219705009478
Energy after perturbing parameters: -2.219644100397


In [8]:
logger.log("Starting QCMPS optimization...")
global_optimizer = SweepOptimizer(logger, log_every_n_steps = 20, max_sweeps=3, local_maxiter = 5, tolerance = 1e-7)
result = global_optimizer.optimize(ansatz, qcmps)

logger.log(f"Final energy: {ansatz.total_energy():.12f}")

Starting QCMPS optimization...
[Optimizer] Sweep 1/3 started.
[Optimizer] Optimizing block 1/8...
[Optimizer] Total Energy: -2.219660915194
[Optimizer] [Sweep 1 Block 1] Total energy: -2.219674150742
[Optimizer] Optimizing block 2/8...
[Optimizer] [Sweep 1 Block 2] Total energy: -2.219736542048
[Optimizer] Optimizing block 3/8...
[Optimizer] [Sweep 1 Block 3] Total energy: -2.219834982564
[Optimizer] Optimizing block 4/8...
[Optimizer] [Sweep 1 Block 4] Total energy: -2.219853766089
[Optimizer] Optimizing block 5/8...
[Optimizer] Total Energy: -2.219860253604
[Optimizer] [Sweep 1 Block 5] Total energy: -2.219933974436
[Optimizer] Optimizing block 6/8...
[Optimizer] [Sweep 1 Block 6] Total energy: -2.219969645637
[Optimizer] Optimizing block 7/8...
[Optimizer] [Sweep 1 Block 7] Total energy: -2.220073855197
[Optimizer] Optimizing block 8/8...
[Optimizer] [Sweep 1 Block 8] Total energy: -2.220139723019
[Optimizer] Optimizing block 7/8...
[Optimizer] Total Energy: -2.220142418362
[Optimiz